# 12 - Paired-bootstrap confidence intervals

**CPU fine. Run all; idempotent.** Reads only saved predictions - no model is
touched - and writes `results/tables/table_bootstrap_ci.csv` plus a JSON with
the same content, which the manuscript's Table 4b is built from.

Protocol: 1,000 resamples (with replacement) of the family-disjoint test
domains. Each resample is applied to **every** model, so differences are
paired; within each resample the three seeds are averaged, so the interval is
on the seed-mean metric the paper reports. Relative FPR@95%TPR reductions are
computed per resample and summarised the same way.

In [ ]:
# --- standard header ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'
if os.path.isdir(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)
else:
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone','-q',f'https://{TOKEN}@{URL}',REPO], check=True)
sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
!pip -q install pyarrow zstandard scikit-learn

In [ ]:
import pandas as pd, numpy as np, json
from pathlib import Path
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
from src.evaluate import predictions, metrics

PRED_DIR = Path(P['artifacts']['predictions']); TAB = Path(P['results']['tables'])
FD = 'family_disjoint_v1'; SEEDS = (42, 43, 44); B = 1000
ROWS = ['a_lexical', 'b_cert', 'e_late', 'd_fused_emb', 'c_fused']

# Align every model to the same domain order (the fused model's test set)
ref = predictions.load(f'fusion_c_fused_{FD}_s42', PRED_DIR).set_index('domain')
doms = ref.index; y = ref['true_label'].values.astype(int); n = len(y)

S = {r: np.stack([predictions.load(f'fusion_{r}_{FD}_s{sd}', PRED_DIR)
                    .set_index('domain').loc[doms, 'raw_score'].values for sd in SEEDS]) for r in ROWS}
TS = np.stack([predictions.load(f'trustscore_{FD}_s{sd}', PRED_DIR)
                 .set_index('domain').loc[doms, 'calibrated_score'].values for sd in SEEDS])
print('test domains:', n, '| positives:', int(y.sum()))

In [ ]:
def fpr95(yb, sb):
    f, t, _ = roc_curve(yb, sb); i = np.searchsorted(t, 0.95, 'left'); return float(f[min(i, len(f)-1)])

rng = np.random.default_rng(42)
acc = {r: {'roc': [], 'fpr': [], 'pr': []} for r in ROWS}
acc['trust'] = {'roc': [], 'fpr': [], 'ece': [], 'brier': []}
rel = {'c_vs_a': [], 'c_vs_b': [], 'c_vs_e': []}

for b in range(B):
    idx = rng.integers(0, n, n); yb = y[idx]
    if yb.min() == yb.max(): continue
    fp = {}
    for r in ROWS:
        acc[r]['roc'].append(np.mean([roc_auc_score(yb, S[r][k][idx]) for k in range(3)]))
        fp[r] = np.mean([fpr95(yb, S[r][k][idx]) for k in range(3)]); acc[r]['fpr'].append(fp[r])
        acc[r]['pr'].append(np.mean([average_precision_score(yb, S[r][k][idx]) for k in range(3)]))
    rel['c_vs_a'].append(1 - fp['c_fused']/fp['a_lexical'])
    rel['c_vs_b'].append(1 - fp['c_fused']/fp['b_cert'])
    rel['c_vs_e'].append(1 - fp['c_fused']/fp['e_late'])
    acc['trust']['roc'].append(np.mean([roc_auc_score(yb, TS[k][idx]) for k in range(3)]))
    acc['trust']['fpr'].append(np.mean([fpr95(yb, TS[k][idx]) for k in range(3)]))
    acc['trust']['ece'].append(np.mean([metrics.expected_calibration_error(yb, TS[k][idx]) for k in range(3)]))
    acc['trust']['brier'].append(np.mean([np.mean((TS[k][idx] - yb)**2) for k in range(3)]))
    if b % 200 == 0: print('resample', b)

ci = lambda v: {'mean': float(np.mean(v)), 'lo': float(np.percentile(v, 2.5)), 'hi': float(np.percentile(v, 97.5))}
res = {r: {m: ci(v) for m, v in d.items()} for r, d in acc.items()}
res['relative_fpr_reduction'] = {k: ci(v) for k, v in rel.items()}

In [ ]:
rows = []
for r, d in res.items():
    for m, c in d.items():
        rows.append({'row': r, 'metric': m, 'mean': round(c['mean'], 4), 'ci_lo': round(c['lo'], 4), 'ci_hi': round(c['hi'], 4)})
tab = pd.DataFrame(rows)
tab.to_csv(TAB/'table_bootstrap_ci.csv', index=False)
(TAB/'table_bootstrap_ci.json').write_text(json.dumps(res, indent=1))
display(tab.pivot(index='row', columns='metric', values='mean').round(4))
print()
for k, c in res['relative_fpr_reduction'].items():
    print(f"{k}: {100*c['mean']:.1f}% relative FPR@95 reduction (95% CI {100*c['lo']:.1f}-{100*c['hi']:.1f}%)")
print('wrote', TAB/'table_bootstrap_ci.csv')

---
The manuscript's Table 4b, the CI clause in the abstract, and the relative
reductions in Section 6.2 are transcribed from `table_bootstrap_ci.csv`.
Re-run this notebook whenever predictions change, then update those cells.